Vitor Hugo da Silva Santos

RA: 625102765

##**Atividade 5:  Uso de n-gramas em NLP**


Exercício: Tome como base o exemplo discutido em aula para o uso de n-gramas (Colab). Execute um código em Python capaz de ler um texto de sua escolha (do seu grupo) e que seja treinado por um modelo de linguagem do tipo MLE para a geração de novas sentenças. Não esqueça de realizar todo o pré-processamento do corpus.

In [1]:
!pip install nltk==3.5
import nltk
import re
import string
import random
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from collections import defaultdict, Counter

nltk.download('punkt')
nltk.download('stopwords')

# ==============================================================================
# 1. CORPUS DE TREINAMENTO (Novo Texto sobre Inteligência Artificial)
# ==============================================================================
CORPUS_TEXT = """
A Inteligência Artificial (IA) é um campo interdisciplinar da ciência da computação que tem como objetivo o desenvolvimento de sistemas capazes de simular processos cognitivos humanos, tais como o raciocínio,
a percepção, o aprendizado e a tomada de decisão. Segundo a Encyclopaedia Britannica (2024), a IA pode ser compreendida como a capacidade de um computador digital ou sistema automatizado de executar tarefas
que normalmente exigem inteligência humana.
Do ponto de vista técnico, a IA baseia-se na aplicação de modelos matemáticos, estatísticos e computacionais que permitem às máquinas analisar dados, identificar padrões e aprimorar seu desempenho de forma
autônoma. Entre as principais abordagens da área destacam-se o Aprendizado de Máquina (Machine Learning), o Aprendizado Profundo (Deep Learning) e o Processamento de Linguagem Natural (PLN),
cada uma voltada para diferentes tipos de problemas e contextos de aplicação.
De modo geral, a IA busca criar sistemas que possam adaptar-se e evoluir com base na experiência, contribuindo para o avanço de diversas áreas, como a automação industrial, a medicina,
a segurança da informação e a análise de dados em larga escala.
"""

# ==============================================================================
# 2. PRÉ-PROCESSAMENTO
# ==============================================================================
def preprocess_corpus(text):
    """Realiza o pré-processamento completo do texto."""

    # 1. Converter para minúsculas
    text_lower = text.lower()

    # 2. Remover pontuações e números
    # Mantendo a pontuação interna (como o hífen em "refere-se") para tokenização
    text_no_punct = re.sub(r'[%s]' % re.escape(string.punctuation.replace('-', '')), '', text_lower)
    text_no_numbers = re.sub(r'\d+', '', text_no_punct)

    # 3. Tokenização em sentenças
    sentences = sent_tokenize(text_no_numbers, language='portuguese')

    tokenized_corpus = []
    for sentence in sentences:
        # 4. Tokenização em palavras
        tokens = word_tokenize(sentence, language='portuguese')

        # 5. Adicionar marcadores de início e fim de sentença (necessário para N-grams)
        if tokens:
            tokenized_corpus.append(['<start>'] + tokens + ['<end>'])

    # Achatar a lista de listas de tokens em uma única lista para o treinamento
    flat_tokens = [token for sentence in tokenized_corpus for token in sentence]

    return flat_tokens

# ==============================================================================
# 3. TREINAMENTO DO MODELO MLE (BIGRAMA)
# ==============================================================================
def train_bigram_mle(tokens):
    """
    Treina um modelo de linguagem Bigrama (N=2) usando Maximum Likelihood Estimation (MLE).
    Retorna um dicionário de probabilidades condicionais P(w_i | w_{i-1}).
    """
    # Contagem de Bigramas (w_{i-1}, w_i)
    bigrams = list(nltk.bigrams(tokens))
    bigram_counts = Counter(bigrams)

    # Contagem de Unigramas (w_{i-1}) - para o denominador da probabilidade
    unigram_counts = Counter(tokens)

    # Cálculo da Probabilidade Condicional P(w_i | w_{i-1})
    # P(w_i | w_{i-1}) = Count(w_{i-1}, w_i) / Count(w_{i-1})
    mle_model = defaultdict(lambda: defaultdict(float))

    for (w_prev, w_curr), count in bigram_counts.items():
        # A probabilidade é a contagem do bigrama dividida pela contagem do unigrama anterior
        mle_model[w_prev][w_curr] = count / unigram_counts[w_prev]

    return mle_model

# ==============================================================================
# 4. GERAÇÃO DE SENTENÇAS
# ==============================================================================
def generate_sentence(model, max_length=30):
    """
    Gera uma nova sentença usando o modelo Bigrama treinado.
    Aumentamos o max_length para 30 para acomodar sentenças mais longas.
    """

    # Começa com o marcador de início de sentença
    current_word = '<start>'
    sentence = []

    for _ in range(max_length):
        # Obtém as probabilidades para a próxima palavra dado o estado atual
        next_word_probs = model[current_word]

        if not next_word_probs:
            # Se não houver transições possíveis, para a geração
            break

        # Extrai as palavras e suas probabilidades
        words = list(next_word_probs.keys())
        probabilities = list(next_word_probs.values())

        # Seleciona a próxima palavra com base na distribuição de probabilidade
        next_word = random.choices(words, weights=probabilities, k=1)[0]

        if next_word == '<end>':
            break

        if next_word != '<start>':
            sentence.append(next_word)

        current_word = next_word

    # Se a sentença atingir o limite de palavras e não terminar com <end>,
    # ela será truncada, mas o max_length foi aumentado para mitigar isso.
    return ' '.join(sentence).capitalize()

# ==============================================================================
# FUNÇÃO PRINCIPAL
# ==============================================================================
def main():
    print("="*50)
    print("MODELO DE LINGUAGEM N-GRAMA (BIGRAMA) - MLE")
    print("Corpus: Novo Texto sobre Inteligência Artificial")
    print("="*50)

    # 1. Pré-processamento
    print("\n[1] Pré-processando o Corpus...")
    tokens = preprocess_corpus(CORPUS_TEXT)
    print(f"Total de tokens processados: {len(tokens)}")

    # 2. Treinamento
    print("\n[2] Treinando o Modelo Bigrama (MLE)...")
    mle_model = train_bigram_mle(tokens)
    print(f"Modelo treinado com {len(mle_model)} palavras únicas como histórico.")

    # 3. Geração de Sentenças
    print("\n[3] Gerando 5 Novas Sentenças:")
    print("-" * 30)
    for i in range(5):
        new_sentence = generate_sentence(mle_model)
        print(f"Sentença {i+1}: {new_sentence}")
    print("-" * 30)

    # 4. Exemplo de Probabilidade (Opcional)
    print("\n[4] Exemplo de Probabilidade (Palavra = 'a'):")
    example_word = 'a'
    if example_word in mle_model:
        print(f"Próximas palavras após '{example_word}':")
        for next_word, prob in sorted(mle_model[example_word].items(), key=lambda item: item[1], reverse=True):
            print(f"  - {next_word}: {prob:.4f}")
    else:
        print(f"A palavra '{example_word}' não foi encontrada no corpus como histórico.")

if __name__ == "__main__":
    main()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for nltk: filename=nltk-3.5-py3-none-any.whl size=1434675 sha256=cb38059462b4ce8509b0c9901f5f8f65b5bff8cd3e190970008d4996494abf1c
  Stored in directory: /root/.cache/pip/wheels/81/c8/97/7683065c88936de3102093c42eed9defe41435ac4cbdedccf5
Successfully built nltk
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
textblob 0.19.0 requires nltk>=3.9, but you have nltk 3.5 which is incompatible.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


MODELO DE LINGUAGEM N-GRAMA (BIGRAMA) - MLE
Corpus: Novo Texto sobre Inteligência Artificial

[1] Pré-processando o Corpus...
Total de tokens processados: 178

[2] Treinando o Modelo Bigrama (MLE)...
Modelo treinado com 118 palavras únicas como histórico.

[3] Gerando 5 Novas Sentenças:
------------------------------
Sentença 1: A ia pode ser compreendida como o raciocínio a segurança da computação que possam adaptar-se e a tomada de problemas e a percepção o aprendizado e computacionais que possam adaptar-se
Sentença 2: A ia pode ser compreendida como a capacidade de máquina machine learning e a automação industrial a percepção o raciocínio a inteligência artificial ia pode ser compreendida como a percepção
Sentença 3: A análise de dados em larga escala
Sentença 4: A medicina a tomada de dados identificar padrões e aprimorar seu desempenho de sistemas capazes de modo geral a segurança da informação e contextos de executar tarefas que tem como
Sentença 5: A tomada de simular processos 

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
